In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

In [2]:
client = bigquery.Client(project="bigquery-prajwals-data")

In [3]:
query = """
SELECT
  transaction_id,
  block_timestamp,
  input_value,
  output_value,
  fee,
  size,
  is_coinbase
FROM `bigquery-prajwals-data.crypto_dw.crypto_sample`
"""
df = client.query(query).to_dataframe()
df.head()

,transaction_id,block_timestamp,input_value,output_value,fee,size,is_coinbase
0,dd31ac3cced479f516a6914a2aacd7b2a1d5d3b77e7157...,2021-01-01 00:16:15+00:00,915664.000000000,914988.000000000,676.000000000,284,False
1,c077188ee4ecaa2f0c1888a0b99e14bab64d672e2f1150...,2021-01-01 00:16:15+00:00,1114929000.000000000,1114872616.000000000,56384.000000000,419,False
2,82330b03678968828f2b35020164e5efea10e3862e06bd...,2021-01-01 00:16:15+00:00,156487975.000000000,156465250.000000000,22725.000000000,417,False
3,f1cc3dc149100a2836eccab3a5118b1216c6ecfd9c1203...,2021-01-01 00:16:15+00:00,12855667.000000000,12676854.000000000,178813.000000000,17144,False
4,2cd749c5626174581b7f6d32f59f46de99ccde839a69de...,2021-01-01 00:16:15+00:00,85887.000000000,83207.000000000,2680.000000000,666,False


In [4]:
df.shape

(15420, 7)

## **DATA CLEANING**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   transaction_id   15420 non-null  object             
 1   block_timestamp  15420 non-null  datetime64[us, UTC]
 2   input_value      15415 non-null  object             
 3   output_value     15420 non-null  object             
 4   fee              15420 non-null  object             
 5   size             15420 non-null  Int64              
 6   is_coinbase      15420 non-null  boolean            
dtypes: Int64(1), boolean(1), datetime64[us, UTC](1), object(4)
memory usage: 768.1+ KB


In [6]:
df = df.drop_duplicates()

In [7]:
df.isna().sum()

transaction_id     0
block_timestamp    0
input_value        5
output_value       0
fee                0
size               0
is_coinbase        0
dtype: int64

In [8]:
df['input_value'].isna().sum()

np.int64(5)

In [9]:
df[df['input_value'].isna()][['is_coinbase']].value_counts()

is_coinbase
True           5
Name: count, dtype: int64

In [10]:
df['input_value'] = df['input_value'].fillna(0)

In [11]:
df['missing_input_value'] = (df['input_value'] == 0).astype(int)

In [12]:
df.isnull().sum()

transaction_id         0
block_timestamp        0
input_value            0
output_value           0
fee                    0
size                   0
is_coinbase            0
missing_input_value    0
dtype: int64

In [13]:
assert (df['fee'] >= 0).all()

assert (df['size'] > 0).all()

assert (df.loc[df['is_coinbase'] == True , 'input_value'] == 0).all()

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   transaction_id       15420 non-null  object             
 1   block_timestamp      15420 non-null  datetime64[us, UTC]
 2   input_value          15420 non-null  object             
 3   output_value         15420 non-null  object             
 4   fee                  15420 non-null  object             
 5   size                 15420 non-null  Int64              
 6   is_coinbase          15420 non-null  boolean            
 7   missing_input_value  15420 non-null  int64              
dtypes: Int64(1), boolean(1), datetime64[us, UTC](1), int64(1), object(4)
memory usage: 888.6+ KB


## **FEATURE ENGINEERING**

In [15]:
df[['input_value', 'output_value', 'fee']].head()

,input_value,output_value,fee
0,915664.000000000,914988.000000000,676.000000000
1,1114929000.000000000,1114872616.000000000,56384.000000000
2,156487975.000000000,156465250.000000000,22725.000000000
3,12855667.000000000,12676854.000000000,178813.000000000
4,85887.000000000,83207.000000000,2680.000000000


In [16]:
SATOSHI = 100_000_000

df['input_btc'] = df['input_value'] / SATOSHI
df['output_btc'] = df['output_value'] / SATOSHI
df['fee_btc'] = df['fee'] / SATOSHI

In [17]:
df['calc_difference'] = df['input_value'] - (df['output_value'] + df['fee'])
df['calc_difference'].describe()

count     15420
unique        6
top        0E-9
freq      15415
Name: calc_difference, dtype: object

In [18]:
df['calc_difference'] = pd.to_numeric(df['calc_difference'])

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   transaction_id       15420 non-null  object             
 1   block_timestamp      15420 non-null  datetime64[us, UTC]
 2   input_value          15420 non-null  object             
 3   output_value         15420 non-null  object             
 4   fee                  15420 non-null  object             
 5   size                 15420 non-null  Int64              
 6   is_coinbase          15420 non-null  boolean            
 7   missing_input_value  15420 non-null  int64              
 8   input_btc            15420 non-null  object             
 9   output_btc           15420 non-null  object             
 10  fee_btc              15420 non-null  object             
 11  calc_difference      15420 non-null  float64            
dtypes: Int64(1), boole

In [20]:
# Fee Rate
df['fee_rate'] = df['fee'] / df['size']

In [21]:
df['fee_rate'] = pd.to_numeric(df['fee_rate'])

In [22]:
# Network Load by Hour
df['hour'] = df['block_timestamp'].dt.hour

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   transaction_id       15420 non-null  object             
 1   block_timestamp      15420 non-null  datetime64[us, UTC]
 2   input_value          15420 non-null  object             
 3   output_value         15420 non-null  object             
 4   fee                  15420 non-null  object             
 5   size                 15420 non-null  Int64              
 6   is_coinbase          15420 non-null  boolean            
 7   missing_input_value  15420 non-null  int64              
 8   input_btc            15420 non-null  object             
 9   output_btc           15420 non-null  object             
 10  fee_btc              15420 non-null  object             
 11  calc_difference      15420 non-null  float64            
 12  fee_rate          

In [24]:
df['input_value'] = pd.to_numeric(df['input_value'])
df['output_value'] = pd.to_numeric(df['output_value'])
df['fee'] = pd.to_numeric(df['fee'])
df['input_btc'] = pd.to_numeric(df['input_btc'])
df['output_btc'] = pd.to_numeric(df['output_btc'])
df['fee_btc'] = pd.to_numeric(df['fee_btc'])

In [25]:
# High Fee Transactions
threshold = df['fee_rate'].quantile(0.95)
df['high_fee_flag'] = (df['fee_rate'] > threshold).astype(int)

In [26]:
# ️Large Value Transactions
large_threshold = df['input_btc'].quantile(0.99)
df['large_tx_flag'] = (df['input_btc'] > large_threshold).astype(int)

In [27]:
# Validation Flag
df['reconsliation_pass'] = (df['calc_difference'].abs() < 0.00001).astype(int)

In [28]:
df['reconsliation_pass'].mean()

np.float64(0.9996757457846952)

In [29]:
df.head()

,transaction_id,block_timestamp,input_value,output_value,fee,size,is_coinbase,missing_input_value,input_btc,output_btc,fee_btc,calc_difference,fee_rate,hour,high_fee_flag,large_tx_flag,reconsliation_pass
0,dd31ac3cced479f516a6914a2aacd7b2a1d5d3b77e7157...,2021-01-01 00:16:15+00:00,9.156640e+05,9.149880e+05,676.0,284,False,0,0.009157,0.009150,0.000007,0.0,2.380282,0,0,0,1
1,c077188ee4ecaa2f0c1888a0b99e14bab64d672e2f1150...,2021-01-01 00:16:15+00:00,1.114929e+09,1.114873e+09,56384.0,419,False,0,11.149290,11.148726,0.000564,0.0,134.568019,0,1,0,1
2,82330b03678968828f2b35020164e5efea10e3862e06bd...,2021-01-01 00:16:15+00:00,1.564880e+08,1.564652e+08,22725.0,417,False,0,1.564880,1.564653,0.000227,0.0,54.496403,0,0,0,1
3,f1cc3dc149100a2836eccab3a5118b1216c6ecfd9c1203...,2021-01-01 00:16:15+00:00,1.285567e+07,1.267685e+07,178813.0,17144,False,0,0.128557,0.126769,0.001788,0.0,10.430063,0,0,0,1
4,2cd749c5626174581b7f6d32f59f46de99ccde839a69de...,2021-01-01 00:16:15+00:00,8.588700e+04,8.320700e+04,2680.0,666,False,0,0.000859,0.000832,0.000027,0.0,4.024024,0,0,0,1


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   transaction_id       15420 non-null  object             
 1   block_timestamp      15420 non-null  datetime64[us, UTC]
 2   input_value          15420 non-null  float64            
 3   output_value         15420 non-null  float64            
 4   fee                  15420 non-null  float64            
 5   size                 15420 non-null  Int64              
 6   is_coinbase          15420 non-null  boolean            
 7   missing_input_value  15420 non-null  int64              
 8   input_btc            15420 non-null  float64            
 9   output_btc           15420 non-null  float64            
 10  fee_btc              15420 non-null  float64            
 11  calc_difference      15420 non-null  float64            
 12  fee_rate          

In [31]:
df['transaction_id'].nunique()
len(df)

15420

In [32]:
df.columns

Index(['transaction_id', 'block_timestamp', 'input_value', 'output_value',
       'fee', 'size', 'is_coinbase', 'missing_input_value', 'input_btc',
       'output_btc', 'fee_btc', 'calc_difference', 'fee_rate', 'hour',
       'high_fee_flag', 'large_tx_flag', 'reconsliation_pass'],
      dtype='object')

In [33]:
df = df.drop_duplicates(subset='transaction_id')

In [34]:
df.isnull().sum().sort_values(ascending=False)

transaction_id         0
block_timestamp        0
input_value            0
output_value           0
fee                    0
size                   0
is_coinbase            0
missing_input_value    0
input_btc              0
output_btc             0
fee_btc                0
calc_difference        0
fee_rate               0
hour                   0
high_fee_flag          0
large_tx_flag          0
reconsliation_pass     0
dtype: int64

In [35]:
df.describe()

,input_value,output_value,fee,size,missing_input_value,input_btc,output_btc,fee_btc,calc_difference,fee_rate,hour,high_fee_flag,large_tx_flag,reconsliation_pass
count,1.542000e+04,1.542000e+04,1.542000e+04,15420.0,15420.000000,15420.000000,15420.000000,15420.000000,1.542000e+04,15420.000000,15420.000000,15420.000000,15420.000000,15420.000000
mean,5.880854e+08,5.882778e+08,2.984579e+04,646.440791,0.000324,5.880854,5.882778,0.000298,-2.222103e+05,55.604955,12.203567,0.050000,0.010052,0.999676
std,2.070516e+10,2.070516e+10,1.660071e+05,4131.517951,0.018005,207.051610,207.051561,0.001660,1.234503e+07,47.041408,6.852480,0.217952,0.099757,0.018005
min,0.000000e+00,0.000000e+00,0.000000e+00,188.0,0.000000,0.000000,0.000000,0.000000,-7.056360e+08,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.086238e+05,4.979285e+05,7.868250e+03,223.0,0.000000,0.005086,0.004979,0.000079,0.000000e+00,24.323961,6.000000,0.000000,0.000000,1.000000
50%,2.476898e+06,2.467486e+06,1.479900e+04,246.0,0.000000,0.024769,0.024675,0.000148,0.000000e+00,52.298123,13.000000,0.000000,0.000000,1.000000
75%,2.297477e+07,2.301955e+07,2.238575e+04,374.0,0.000000,0.229748,0.230196,0.000224,0.000000e+00,76.345291,18.000000,0.000000,0.000000,1.000000
max,1.313166e+12,1.313166e+12,8.660174e+06,185394.0,1.000000,13131.663921,13131.662921,0.086602,0.000000e+00,1767.000000,23.000000,1.000000,1.000000,1.000000


## **POSTGRESQL CONNECTION**

In [40]:
# # Postgresql Connection
from sqlalchemy import create_engine

username = "postgres"
password = "25012004"
host = "localhost"
port = "5433"
database = "Bitcoin Fee Analytics"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = 'bitcoin_data'
df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"DF loaded succesfully in database '{database}' in table '{table_name}'.")

DF loaded succesfully in database 'Bitcoin Fee Analytics' in table 'bitcoin_data'.
